# Food Demand Analysis

Notebook reproducibly merges the source files, validates data quality, computes metrics/tests, and creates charts. Run from the project root.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, spearmanr
ROOT = Path.cwd()
if not (ROOT / "data/raw/train.csv").exists():
    ROOT = ROOT.parent
RAW = ROOT / "data/raw"
OUT = ROOT / "reports"
FIG = OUT / "figures"
OUT.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)


In [2]:
train = pd.read_csv(RAW / "train.csv")
test = pd.read_csv(RAW / "test.csv")
meal = pd.read_csv(RAW / "meal_info.csv")
center = pd.read_csv(RAW / "fulfilment_center_info.csv")

print("Raw shapes:", {"train": train.shape, "test": test.shape, "meal": meal.shape, "center": center.shape})
print("Training order sum before merge:", int(train["num_orders"].sum()))


Raw shapes: {'train': (456548, 9), 'test': (32573, 8), 'meal': (51, 3), 'center': (77, 5)}
Training order sum before merge: 119557485


In [3]:
df = train.merge(meal, on="meal_id", how="left", validate="many_to_one")
print("After meal merge:", df.shape, "unmatched meal IDs:", int(df["category"].isna().sum()))
df = df.merge(center, on="center_id", how="left", validate="many_to_one")
print("After center merge:", df.shape, "unmatched centers:", int(df["center_type"].isna().sum()))
assert len(df) == len(train), "Merge unexpectedly changed row count"
assert int(df["num_orders"].sum()) == int(train["num_orders"].sum()), "Merge changed total orders"
df["revenue"] = df["checkout_price"] * df["num_orders"]
df["discount_rate"] = (df["base_price"] - df["checkout_price"]) / df["base_price"].replace(0, np.nan)
df["any_promotion"] = ((df["emailer_for_promotion"] == 1) | (df["homepage_featured"] == 1)).astype(int)


After meal merge: (456548, 11) unmatched meal IDs: 0
After center merge: (456548, 15) unmatched centers: 0


In [4]:
quality = pd.DataFrame([
    ["Missing cells in train", int(train.isna().sum().sum()), "Inspect columns; avoid blind imputation"],
    ["Exact duplicate rows", int(train.duplicated().sum()), "Review before removing"],
    ["Checkout price above base price", int((train.checkout_price > train.base_price).sum()), "Retain signed discount unless confirmed erroneous"],
    ["Rows with nonpositive orders", int((train.num_orders <= 0).sum()), "Investigate if present"],
], columns=["issue", "count", "decision"])
display(quality)


,issue,count,decision
0,Missing cells in train,0,Inspect columns; avoid blind imputation
1,Exact duplicate rows,0,Review before removing
2,Checkout price above base price,116101,Retain signed discount unless confirmed erroneous
3,Rows with nonpositive orders,0,Investigate if present


In [5]:
weekly = df.groupby("week", as_index=False).agg(total_orders=("num_orders","sum"), revenue=("revenue","sum"))
category_metrics = df.groupby("category", as_index=False).agg(total_orders=("num_orders","sum"), revenue=("revenue","sum")).sort_values("total_orders", ascending=False)
category_metrics["share_pct"] = category_metrics.total_orders / df.num_orders.sum() * 100
category_metrics["cumulative_share_pct"] = category_metrics.share_pct.cumsum()
meal_metrics = df.groupby(["meal_id","category","cuisine"], as_index=False).agg(total_orders=("num_orders","sum"), revenue=("revenue","sum")).sort_values("total_orders", ascending=False)
center_metrics = df.groupby(["center_id","center_type","op_area"], as_index=False).agg(total_orders=("num_orders","sum"), revenue=("revenue","sum"), mean_orders_per_record=("num_orders","mean"), records=("num_orders","size")).sort_values("total_orders", ascending=False)
center_type_metrics = df.groupby("center_type", as_index=False).agg(total_orders=("num_orders","sum"), revenue=("revenue","sum"), mean_orders_per_record=("num_orders","mean"), centers=("center_id","nunique"))
display(weekly.head(), category_metrics.head(), meal_metrics.head(), center_metrics.head(), center_type_metrics)


,week,total_orders,revenue
0,1,792261,2.185165e+08
1,2,787084,1.999142e+08
2,3,695262,1.780359e+08
3,4,743529,1.961360e+08
4,5,1198675,2.596449e+08


,category,total_orders,revenue,share_pct,cumulative_share_pct
0,Beverages,40480525,7.347304e+09,33.858629,33.858629
8,Rice Bowl,20874063,5.881531e+09,17.459436,51.318065
10,Sandwich,17636782,4.663427e+09,14.751717,66.069782
9,Salad,10944336,3.032256e+09,9.154037,75.223819
7,Pizza,7383720,3.955956e+09,6.175874,81.399693


,meal_id,category,cuisine,total_orders,revenue
30,2290,Rice Bowl,Indian,9770303,2.445550e+09
22,1885,Beverages,Thai,8346246,1.098536e+09
16,1754,Sandwich,Italian,6818059,1.752753e+09
26,1993,Beverages,Thai,6656856,8.675727e+08
1,1109,Rice Bowl,Indian,6367773,1.659226e+09


,center_id,center_type,op_area,total_orders,revenue,mean_orders_per_record,records
2,13,TYPE_B,6.7,4296545,1.127045e+09,609.784984,7046
18,43,TYPE_A,5.1,3920294,9.864821e+08,562.452511,6970
0,10,TYPE_B,6.3,3381018,9.287786e+08,481.969779,7015
21,52,TYPE_B,5.6,3125835,8.095873e+08,446.994852,6993
63,137,TYPE_A,4.4,3117478,7.729365e+08,452.201625,6894


,center_type,total_orders,revenue,mean_orders_per_record,centers
0,TYPE_A,68978517,1.811730e+10,262.394456,43
1,TYPE_B,29996073,8.106742e+09,318.856145,15
2,TYPE_C,20582895,5.698733e+09,206.670097,19


In [6]:
def promotion_test(flag):
    key_counts = df.groupby(["center_id","meal_id"])[flag].nunique()
    keys = key_counts[key_counts == 2].index
    matched = df.set_index(["center_id","meal_id"]).loc[keys].reset_index()
    yes = matched.loc[matched[flag] == 1, "num_orders"]
    no = matched.loc[matched[flag] == 0, "num_orders"]
    stat, p = mannwhitneyu(yes, no, alternative="two-sided")
    return {"flag": flag, "matched_pairs": len(keys), "n_yes": len(yes), "n_no": len(no),
            "mean_yes": yes.mean(), "mean_no": no.mean(),
            "relative_difference_pct": (yes.mean()/no.mean()-1)*100,
            "U": stat, "p_value": p}
promotion_results = pd.DataFrame([promotion_test("emailer_for_promotion"), promotion_test("homepage_featured")])
display(promotion_results)
rho, p_rho = spearmanr(df["discount_rate"].dropna(), df.loc[df.discount_rate.notna(), "num_orders"])
print(f"Spearman discount/order: rho={rho:.4f}, p={p_rho:.4g}")


,flag,matched_pairs,n_yes,n_no,mean_yes,mean_no,relative_difference_pct,U,p_value
0,emailer_for_promotion,2830,37050,338694,631.097544,254.642539,147.836653,8.740582e+09,0.0
1,homepage_featured,2928,49855,338548,594.884786,241.372937,146.458776,1.169418e+10,0.0


Spearman discount/order: rho=0.1218, p=0


In [7]:
meal_week = df.groupby(["meal_id","week"], as_index=False).agg(weekly_orders=("num_orders","sum"))
volatility = meal_week.groupby("meal_id").weekly_orders.agg(mean_weekly_orders="mean", std_weekly_orders="std").reset_index()
volatility["cv"] = volatility.std_weekly_orders / volatility.mean_weekly_orders
volatility = volatility.merge(meal[["meal_id","category","cuisine"]], on="meal_id", how="left")
volatility["total_orders"] = volatility.meal_id.map(df.groupby("meal_id").num_orders.sum())
volatility["volume_rank"] = volatility.total_orders.rank(ascending=False, method="min")
volatility["cv_rank"] = volatility.cv.rank(ascending=False, method="min")
volatility["priority_score"] = volatility.volume_rank + volatility.cv_rank
volatility = volatility.sort_values("priority_score")
display(volatility.head(15))


,meal_id,mean_weekly_orders,std_weekly_orders,cv,category,cuisine,total_orders,volume_rank,cv_rank,priority_score
30,2290,67381.400000,60641.916250,0.899980,Rice Bowl,Indian,9770303,1.0,6.0,7.0
25,1971,39507.724138,43605.509263,1.103721,Sandwich,Italian,5728620,7.0,2.0,9.0
16,1754,47021.096552,41698.021104,0.886794,Sandwich,Italian,6818059,3.0,7.0,10.0
35,2490,29673.000000,27959.324242,0.942248,Salad,Italian,3709125,12.0,5.0,17.0
15,1727,32661.979310,26274.639486,0.804441,Rice Bowl,Indian,4735987,9.0,9.0,18.0
1,1109,43915.675862,24620.199675,0.560624,Rice Bowl,Indian,6367773,5.0,16.0,21.0
24,1962,22315.724138,15268.032510,0.684183,Pizza,Continental,3235780,15.0,13.0,28.0
41,2581,16544.827586,11934.251022,0.721328,Pizza,Continental,2399000,17.0,12.0,29.0
13,1558,12061.655172,9383.111367,0.777929,Pizza,Continental,1748940,22.0,10.0,32.0
48,2826,35104.158621,17059.037946,0.485955,Sandwich,Italian,5090103,8.0,24.0,32.0


In [ ]:
# Finding-led charts
plt.figure(figsize=(10,5)); plt.plot(weekly.week, weekly.total_orders)
plt.title("Weekly demand varies substantially across training weeks"); plt.xlabel("Week"); plt.ylabel("Orders"); plt.tight_layout()
plt.savefig(FIG/"weekly_demand.png", dpi=160); plt.show()

top = category_metrics.head(10).sort_values("total_orders")
plt.figure(figsize=(10,5)); plt.barh(top.category, top.total_orders)
plt.title("Beverages and Rice Bowl lead category demand"); plt.xlabel("Orders"); plt.tight_layout()
plt.savefig(FIG/"category_demand.png", dpi=160); plt.show()

sub = df.sample(min(15000, len(df)), random_state=42)
plt.figure(figsize=(8,5)); plt.scatter(sub.discount_rate*100, sub.num_orders, alpha=.25, s=10)
plt.title(f"Discount has weak positive rank association with demand (rho={rho:.2f})")
plt.xlabel("Discount rate (%)"); plt.ylabel("Orders"); plt.tight_layout()
plt.savefig(FIG/"discount_vs_demand.png", dpi=160); plt.show()

vals = [df.loc[df.emailer_for_promotion==0,"num_orders"].sample(min(50000,(df.emailer_for_promotion==0).sum()), random_state=42),
        df.loc[df.emailer_for_promotion==1,"num_orders"].sample(min(50000,(df.emailer_for_promotion==1).sum()), random_state=42)]
plt.figure(figsize=(8, 5))

vals = [
    df.loc[df.emailer_for_promotion == 0, "num_orders"]
      .sample(
          n=min(50000, (df.emailer_for_promotion == 0).sum()),
          random_state=42
      ),
    df.loc[df.emailer_for_promotion == 1, "num_orders"]
      .sample(
          n=min(50000, (df.emailer_for_promotion == 1).sum()),
          random_state=42
      )
]

plt.boxplot(
    vals,
    tick_labels=["No email promotion", "Email promotion"],
    showfliers=False
)

plt.title("Orders are higher in rows with email promotion")
plt.ylabel("Orders")
plt.tight_layout()

plt.savefig(FIG / "email_promotion_boxplot.png", dpi=160)
plt.show()

SyntaxError: invalid syntax (3204492743.py, line 45)

In [9]:
# Export reproducible outputs
weekly.to_csv(OUT/"weekly_metrics.csv", index=False)
category_metrics.to_csv(OUT/"category_metrics.csv", index=False)
meal_metrics.to_csv(OUT/"meal_metrics.csv", index=False)
center_metrics.to_csv(OUT/"center_metrics.csv", index=False)
center_type_metrics.to_csv(OUT/"center_type_metrics.csv", index=False)
promotion_results.to_csv(OUT/"promotion_tests.csv", index=False)
volatility.to_csv(OUT/"meal_variability.csv", index=False)
quality.to_csv(ROOT/"docs/quality_checks.csv", index=False)
print("Exports saved under reports/ and docs/.")


Exports saved under reports/ and docs/.
